In [6]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import sentencepiece
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
eg=pd.read_parquet("/egy.parquet")
sm1=pd.read_json("/sumarabic_test.jsonl", lines=True)
sm2=pd.read_json("/sumarabic_train.jsonl", lines=True)
sm3=pd.read_json("/sumarabic_valid.jsonl", lines=True)
smds=pd.read_csv("/summarizedd.csv")

In [9]:
AraSum = pd.read_csv("/ArabicSum.csv", sep = "\t" , engine = "python" , header = None , names = ["index" , "text" , "summarized"])
AraSum = AraSum.drop("index" , axis = 1)

In [10]:
sm_train=sm2[["text","headline"]]
sm_train=sm_train.rename(columns={"headline":"summarized"})

In [11]:
sm_test=sm1[["text","headline"]]
sm_test=sm_test.rename(columns={"headline":"summarized"})

In [12]:
sm_valid=sm3[["text","headline"]]
sm_valid=sm_valid.rename(columns={"headline":"summarized"})

In [13]:
eg.head()

,text,summarized_text,source_topics
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...,تأثير الأعباء الاقتصادية على قرارات الإنجاب


In [14]:
eg=eg[["text","summarized_text"]].rename(columns={"summarized_text":"summarized"})

In [15]:
smds.head()

,text,type,Processed Text,summarizer
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,culture,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",culture,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,culture,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,culture,شهدت برلين الجمعه افتتاح مسجد فريد نوعه الاقل ...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,culture,نعت وزاره المنشد عز بن محمود انتقل جوار يوم تن...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...


In [16]:
smds=smds[["text","summarizer"]].rename(columns={"summarizer":"summarized"})

In [17]:
AraSum.head()

,text,summarized
0,"حقق حزب ""البديل من أجل ألمانيا"" اليميني الشعبو...",تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...
1,بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...,بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...
2,قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...,أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...
3,أعلن مسؤول في البنتاغون أن جنودا أمريكيين في ش...,قالت وزارة الدفاع الأمريكية (البنتاغون) إن داع...
4,اجتمعت المجموعة الدولية لدعم سوريا على هامش اج...,فشلت الولايات المتحدة وروسيا في الاتفاق على كي...


In [18]:
concat_df=pd.concat([eg,smds,AraSum],ignore_index=True)

In [19]:
concat_df.head()

,text,summarized
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...


In [20]:
from sklearn.model_selection import train_test_split
concat_df_train,concat_df_test = train_test_split( concat_df,test_size=0.15, random_state=42
)
concat_df_train2,concat_df_validat = train_test_split( concat_df,test_size=0.15, random_state=42
)

In [21]:
concat_df_train2

,text,summarized
10453,\nقال الناشط الحقوقي والمهتم بالشأن الليبي مصط...,\nقال الناشط الحقوقي والمهتم بالشأن الليبي مصط...
40384,تقول الحكومة المصرية إن مشكلة الكلاب الضالة با...,"""ضد تسميم الكلاب"" و""ضد تصدير الكلاب"" و""لا لانت..."
45160,حكمت محكمة ألمانية اليوم الجمعة (10 شباط/ فبرا...,بعد مرور عام تقريباً على الهجوم الذي استهدف جن...
8528,\nنشرت بلدية تونس العاصمة صورا لتسماح بحديقة ا...,"وأكدت المكلفة بالاعلام ببلدية تونس للجوهرة ""اف..."
38154,أظهر استطلاع للرأي اليوم الخميس (13 حزيران/ يو...,"في أحدث استطلاع رأي بتكليف من منظمة ""السلام ال..."
...,...,...
54343,شهدت الساعات القليلة الماضية توتراً حاداً بين ...,اتفق الرئيسان التركي والأمريكي على إقامة منطقة...
38158,"""أزمة في بايرن""، خبر معتاد قادم من عرين النادي...",أزمة في بايرن أم لا؟ بعد تعادلين وخسارة بدأ ال...
860,قيادة الاستدامة في إدارة المخلفات بتبدأ بوضع ا...,وضع استراتيجية مستدامة لإدارة المخلفات بيركز ع...
15795,وجه البابا فرنسيس اليوم الأحد (25 مايو/ أيار 2...,دعا البابا فرنسيس الرئيسين الفلسطيني والإسرائي...


In [22]:
final_df_train=pd.concat([sm_train,concat_df_train2],ignore_index=True)
final_df_test=pd.concat([sm_test,concat_df_test],ignore_index=True)
final_df_valid=pd.concat([sm_valid,concat_df_validat],ignore_index=True)

In [23]:
final_df_train.head()

,text,summarized
0,اختتمت مساء أول من أمس نهائيات بطولة الإمارات ...,المصري فؤاد الطاهر بطل للشطرنج الديناميكي
1,مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب)...,"حملة انتخابية محمومة قبل ""الثلاثاء الكبير"""
2,أفاد مصدر في شرطة أبوظبي بأن نحو 700 طفل يتواف...,55% من نزلاء «الأحداث» مواطنون
3,قال استشاري أمراض الكلى في مدينة الشيخ خليفة ا...,مركز متخصص في الكلى بمستشفى خليفة
4,بدأت القيادة العامة لشرطة أبوظبي التنسيق لإنشا...,تأهيل ضحايا حوادث المرور في أبوظبي


In [24]:
import re

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    #Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    #Remove emails
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)

    #Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    #Remove script/style blocks
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.IGNORECASE | re.DOTALL)

    #Remove CSS styles
    text = re.sub(r'<style.*?>.*?</style>', ' ', text, flags=re.IGNORECASE | re.DOTALL)

    #Remove mentions
    text = re.sub(r'@\w+', ' ', text)

    #Remove hashtag symbol but keep word
    text = re.sub(r'#(\w+)', r'\1', text)

    #Remove tatweel
    text = re.sub(r'ـ+', '', text)

    #Remove English letters
    text = re.sub(r'[a-zA-Z]', '', text)

    #Remove parentheses and slashes
    text = re.sub(r'[()/]', ' ', text)

    #Remove single isolated Arabic letters
    text = re.sub(r'\s[ء-ي]\s', ' ', text)

    #Keep Arabic + punctuation only
    text = re.sub(r'[^\u0600-\u06FF.,!?؟،:؛()\-\n\s]', ' ', text)

    #Normalize punctuation
    text = text.replace('…', '.')
    text = text.replace('٪', '%')
    text = text.replace('―', '-')

    #Normalize spaces
    text = re.sub(r'[ \t]+', ' ', text)

    #Normalize newlines
    text = re.sub(r'\n+', '\n', text)

    return text.strip()

In [25]:
final_df_train["clean_text"] =final_df_train["text"].apply(clean_arabic_text)
final_df_train["clean_summary"]= final_df_train["summarized"].apply(clean_arabic_text)

final_df_test["clean_text"] = final_df_test["text"].apply(clean_arabic_text)
final_df_test["clean_summary"]= final_df_test["summarized"].apply(clean_arabic_text)

final_df_valid["clean_text"] = final_df_valid["text"].apply(clean_arabic_text)
final_df_valid["clean_summary"]= final_df_valid["summarized"].apply(clean_arabic_text)

In [26]:
pip install transformers sentencepiece

In [27]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/AraT5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

In [28]:
def tkn(row):
    model_inputs = tokenizer(
        "summarize: " + row["clean_text"],
        max_length=512,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=row["clean_summary"],
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    label_ids = [
        l if l != tokenizer.pad_token_id else -100
        for l in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

In [29]:
df_final_train_tokenized = final_df_train.apply(tkn, axis=1)

In [30]:
df_final_train_tokenized

,0
0,"[input_ids, attention_mask, labels]"
1,"[input_ids, attention_mask, labels]"
2,"[input_ids, attention_mask, labels]"
3,"[input_ids, attention_mask, labels]"
4,"[input_ids, attention_mask, labels]"
...,...
54769,"[input_ids, attention_mask, labels]"
54770,"[input_ids, attention_mask, labels]"
54771,"[input_ids, attention_mask, labels]"
54772,"[input_ids, attention_mask, labels]"


In [31]:
sample = df_final_train_tokenized.iloc[0]

print(sample["input_ids"][:20])
print(sample["labels"][:20])

print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
print(tokenizer.decode(
    [t for t in sample["labels"] if t != -100],
    skip_special_tokens=True
))

[35880, 4633, 16410, 33, 33261, 527, 416, 6, 397, 13239, 1262, 1071, 97891, 468, 6309, 31, 20, 15068, 53, 40977]
[920, 5451, 16820, 1992, 97891, 468, 6309, 31, 20, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
summarize: اختتمت مساء أول من أمس نهائيات بطولة الإمارات للشطرنج الديناميكي المفتوحة التي نظمها اتحاد الإمارات للشطرنج بمقر نادي دبي للشطرنج والثقافة حيث توج الأستاذ الدولي المصري فؤاد الطاهر بطلا للمسابقة برصيد نقاط من جولات.
المصري فؤاد الطاهر بطل للشطرنج الديناميكي


In [32]:
sample = df_final_train_tokenized.iloc[0]

In [33]:
sample

{'input_ids': [35880, 4633, 16410, 33, 33261, 527, 416, 6, 397, 13239, 1262, 1071, 97891, 468, 6309, 31, 20, 15068, 53, 40977, 1603, 1071, 97891, 7220, 739, 1106, 97891, 11625, 178, 16724, 3643, 675, 920, 5451, 16820, 14756, 58487, 5050, 1904, 6, 17012, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Phase 2 – Model Design, Implementation, Training & Evaluation

Fine-tuning **AraT5-base** (UBC-NLP) for abstractive Arabic summarization.

In [34]:
!pip install -q transformers[torch] datasets rouge-score sentencepiece accelerate


  Preparing metadata (setup.py) ... done


In [35]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'None'}")


Device: cuda
GPU: NVIDIA A100-SXM4-40GB


### 2.1 – PyTorch Dataset Wrapper

In [36]:
class ArabicSumDataset(Dataset):
    def __init__(self, tokenized_series):
        self.data = tokenized_series.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        return {
            "input_ids":      torch.tensor(row["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(row["attention_mask"], dtype=torch.long),
            "labels":         torch.tensor(row["labels"],         dtype=torch.long),
        }


df_final_valid_tokenized = final_df_valid.apply(tkn, axis=1)
df_final_test_tokenized  = final_df_test.apply(tkn,  axis=1)

train_dataset = ArabicSumDataset(df_final_train_tokenized)
valid_dataset = ArabicSumDataset(df_final_valid_tokenized)
test_dataset  = ArabicSumDataset(df_final_test_tokenized)

print(f"Train: {len(train_dataset):,} | Valid: {len(valid_dataset):,} | Test: {len(test_dataset):,}")


Train: 54,774 | Valid: 9,387 | Test: 9,389


### 2.2 – Load AraT5-base for Seq2Seq

In [38]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()


NameError: name 'model' is not defined

In [39]:
MODEL_NAME = "UBC-NLP/AraT5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, force_download=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, force_download=True)
model.to(DEVICE)
print("All params clean:", all(not torch.isnan(p).any() for _, p in model.named_parameters()))

config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

All params clean: True


### 2.3 – Data Collator

In [40]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)


### 2.4 – ROUGE Evaluation Metric

Computed after each epoch on the validation set (ROUGE-1, ROUGE-2, ROUGE-L F1).

In [41]:
import re

def normalize_arabic(text):
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def compute_rouge_manually(pred, ref):
    pred_tokens = pred.split()
    ref_tokens  = ref.split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0, 0.0, 0.0

    # ROUGE-1
    pred_set = set(pred_tokens)
    ref_set  = set(ref_tokens)
    overlap1 = len(pred_set & ref_set)
    r1 = (2 * overlap1 / (len(pred_tokens) + len(ref_tokens))) if overlap1 > 0 else 0.0

    # ROUGE-2
    pred_bigrams = set(zip(pred_tokens, pred_tokens[1:]))
    ref_bigrams  = set(zip(ref_tokens,  ref_tokens[1:]))
    overlap2 = len(pred_bigrams & ref_bigrams)
    r2 = (2 * overlap2 / (len(pred_bigrams) + len(ref_bigrams))) if overlap2 > 0 else 0.0

    # ROUGE-L (LCS)
    m, n = len(ref_tokens), len(pred_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == pred_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    rl = (2 * lcs / (m + n)) if lcs > 0 else 0.0

    return r1, r2, rl


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1)

    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,      skip_special_tokens=True)

    decoded_preds  = [normalize_arabic(p) for p in decoded_preds]
    decoded_labels = [normalize_arabic(r) for r in decoded_labels]

    r1, r2, rl = [], [], []
    for pred, ref in zip(decoded_preds, decoded_labels):
        s1, s2, sl = compute_rouge_manually(pred, ref)
        r1.append(s1)
        r2.append(s2)
        rl.append(sl)

    return {
        "rouge1": round(np.mean(r1) * 100, 2),
        "rouge2": round(np.mean(r2) * 100, 2),
        "rougeL": round(np.mean(rl) * 100, 2),
    }

### 2.5 – Training Arguments & Trainer

>

In [42]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

OUTPUT_DIR = "/content/anat5-arabic-summarization"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    warmup_steps=1000,          # increased from 500
    weight_decay=0.01,
    learning_rate=3e-5,         # lowered from 5e-5
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    fp16=False,
    bf16=True,
    seed=42,
)

model.gradient_checkpointing_enable()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer ready.")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready.


In [43]:
import torch
from torch.utils.data import DataLoader

loader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)
batch = next(iter(loader))
batch = {k: v.to(DEVICE) for k, v in batch.items()}

print("input_ids has nan:", torch.isnan(batch["input_ids"].float()).any())
print("labels has nan:", torch.isnan(batch["labels"].float()).any())
print("input_ids max:", batch["input_ids"].max().item())
print("labels max:", batch["labels"].max().item())
print("vocab size:", tokenizer.vocab_size)
print("model vocab size:", model.config.vocab_size)

input_ids has nan: tensor(False, device='cuda:0')
labels has nan: tensor(False, device='cuda:0')
input_ids max: 97891
labels max: 100786
vocab size: 110000
model vocab size: 110080


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


In [44]:
model.eval()
with torch.no_grad():
    batch = next(iter(torch.utils.data.DataLoader(valid_dataset, batch_size=8, collate_fn=data_collator)))
    input_ids = batch["input_ids"].to(DEVICE)
    labels    = batch["labels"]

    generated = model.generate(input_ids, max_new_tokens=128, num_beams=4)

    # Pad generated to same length for compute_metrics
    max_len = 128
    preds = np.zeros((len(generated), max_len), dtype=np.int64)
    for i, g in enumerate(generated):
        g = g.cpu().numpy()
        preds[i, :len(g)] = g[:max_len]

    labels_np = labels.numpy()

    # Call compute_metrics directly
    result = compute_metrics((preds, labels_np))
    print("ROUGE scores:", result)

    # Also print a few examples to verify
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(
        np.where(labels_np != -100, labels_np, tokenizer.pad_token_id),
        skip_special_tokens=True
    )
    for p, r in zip(decoded_preds[:3], decoded_labels[:3]):
        print(f"Pred : {p}")
        print(f"Ref  : {r}")
        print()

ROUGE scores: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0)}
Pred : 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫
Ref  : اطلاق سراح معارض سوري اعتقل السبت

Pred : 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫
Ref  : الساسة البريطانيون ينهبون المال العام

Pred : 🚫 Malay Malay Malay Malay Malay Malay Malay Malay Malay Malay Malay 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫 🚫

### 2.6 – Train

In [ ]:


train_result = trainer.train()
print(train_result.metrics)


Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")


### 2.7 – Training & Validation Loss Curves

In [ ]:
log_history = trainer.state.log_history

train_loss = [(e["epoch"], e["loss"])     for e in log_history if "loss"      in e and "eval_loss" not in e]
eval_loss  = [(e["epoch"], e["eval_loss"]) for e in log_history if "eval_loss" in e]
rouge_l    = [(e["epoch"], e["eval_rougeL"]) for e in log_history if "eval_rougeL" in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(*zip(*train_loss), label="Train Loss")
axes[0].plot(*zip(*eval_loss),  label="Val Loss", linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves"); axes[0].legend()

if rouge_l:
    axes[1].plot(*zip(*rouge_l), color="green", marker="o", label="ROUGE-L")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("ROUGE-L F1")
    axes[1].set_title("ROUGE-L per Epoch"); axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


### 2.8 – Final Evaluation on Test Set

In [ ]:
test_results = trainer.predict(test_dataset)

test_metrics = compute_metrics(
    (test_results.predictions, test_results.label_ids)
)

print("Test Set ROUGE Scores:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")


### 2.9 – Qualitative Analysis: Predicted vs. Gold Summaries

In [ ]:
def summarize(text, max_input=512, max_new_tokens=128, num_beams=4):
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=max_input,
        truncation=True,
    ).to(DEVICE)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


samples = final_df_test[["clean_text", "clean_summary"]].dropna().sample(5, random_state=42)

for i, (_, row) in enumerate(samples.iterrows(), 1):
    pred = summarize(row["clean_text"])
    sep = "=" * 60
    print(sep)
    print(f"Example {i}")
    print(f"Source   : {row['clean_text'][:200]}...")
    print(f"Reference: {row['clean_summary']}")
    print(f"Predicted: {pred}")
